<a href="https://colab.research.google.com/github/swrobuts/dav/blob/main/notebooks/05_Daten_Bereinigen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧹 Notebook 05: Daten bereinigen

**Szenario**: Du bekommst `verkaufsdaten_mit_problemen.xlsx` – ein typischer Datensatz aus der Praxis mit allen klassischen Problemen.

**Lernziele:**
- Fehlende Werte erkennen und behandeln (dropna, fillna, Imputation)
- Duplikate entfernen
- Strings normalisieren
- Datentypen korrigieren
- Ausreißer behandeln

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Datensatz mit Problemen laden
df = pd.read_excel(BASE_URL + "verkaufsdaten_mit_problemen.xlsx")
print(f'Shape: {df.shape}')
print(f'Spalten: {list(df.columns)}')
print()
print(df.head(10))

## 1️⃣ Datenqualität beurteilen

In [ ]:
print('=== Datenqualitäts-Check ===')
print()
print('Datentypen:')
print(df.dtypes)
print()
print('Fehlende Werte:')
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0].to_string())
    print(f'Gesamt: {missing.sum()} fehlende Werte ({missing.sum()/df.size*100:.1f}%)')
else:
    print('Keine fehlenden Werte gefunden.')
print()
print(f'Duplikate: {df.duplicated().sum()}')
print()
print('Statistische Übersicht:')
print(df.describe())

## 2️⃣ Fehlende Werte behandeln

In [ ]:
df_clean = df.copy()

# Strategie je nach Spaltentyp
for col in df_clean.columns:
    null_count = df_clean[col].isnull().sum()
    if null_count == 0:
        continue

    pct = null_count / len(df_clean) * 100
    dtype = df_clean[col].dtype

    print(f'{col}: {null_count} fehlend ({pct:.1f}%)')

    if pct > 50:
        print(f'  → Spalte hat >50% fehlend – evtl. weglassen!')
    elif dtype in ['float64', 'int64']:
        # Median für numerische Spalten
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f'  → Mit Median ({median_val:.2f}) aufgefüllt')
    else:
        # Modus für kategorische
        if df_clean[col].mode().empty:
            df_clean[col] = df_clean[col].fillna('Unbekannt')
        else:
            mode_val = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(mode_val)
        print(f'  → Mit Modus aufgefüllt')

print(f'\nNach Bereinigung: {df_clean.isnull().sum().sum()} fehlende Werte')

## 3️⃣ Duplikate entfernen

In [ ]:
n_before = len(df_clean)
duplicates = df_clean.duplicated().sum()
print(f'Duplikate gefunden: {duplicates}')

if duplicates > 0:
    print('Duplikat-Beispiel:')
    print(df_clean[df_clean.duplicated(keep=False)].head(4))
    df_clean = df_clean.drop_duplicates()
    print(f'\nEntfernt: {n_before - len(df_clean)} Duplikate')
else:
    print('Keine Duplikate vorhanden.')

print(f'Datensatz jetzt: {len(df_clean)} Zeilen')

## 4️⃣ String-Bereinigung

In [ ]:
# String-Spalten bereinigen
for col in df_clean.select_dtypes(include='object').columns:
    # Leerzeichen entfernen
    df_clean[col] = df_clean[col].str.strip()
    # Einheitliche Groß-/Kleinschreibung
    df_clean[col] = df_clean[col].str.title()

    # Eindeutige Werte zeigen
    unique_vals = df_clean[col].unique()
    print(f'{col}: {len(unique_vals)} eindeutige Werte')
    if len(unique_vals) <= 10:
        print(f'  Werte: {sorted([str(v) for v in unique_vals])}')

print('\n✅ String-Bereinigung abgeschlossen!')

## 5️⃣ Datentypen korrigieren

In [ ]:
print('Datentypen vorher:')
print(df_clean.dtypes)
print()

# Versuche numerische Spalten zu konvertieren
for col in df_clean.columns:
    if df_clean[col].dtype == 'object':
        # Versuche Datum
        try:
            df_clean[col] = pd.to_datetime(df_clean[col], dayfirst=True, errors='raise')
            print(f'{col}: → datetime64')
            continue
        except:
            pass
        # Versuche Zahl
        numeric_series = pd.to_numeric(df_clean[col].str.replace(',', '.'), errors='coerce')
        if numeric_series.notna().sum() / len(df_clean) > 0.8:
            df_clean[col] = numeric_series
            print(f'{col}: → numeric')

print()
print('Datentypen nachher:')
print(df_clean.dtypes)

## 6️⃣ Vorher/Nachher Qualitätsscore

In [ ]:
def quality_score(df):
    completeness = 1 - df.isnull().sum().sum() / df.size
    uniqueness = 1 - df.duplicated().sum() / len(df)
    return {'Vollständigkeit': round(completeness * 100, 1),
            'Eindeutigkeit': round(uniqueness * 100, 1),
            'Gesamt': round((completeness + uniqueness) / 2 * 100, 1)}

original = pd.read_excel(BASE_URL + "verkaufsdaten_mit_problemen.xlsx")
before = quality_score(original)
after = quality_score(df_clean)

print('=== Qualitätsscore: Vorher vs. Nachher ===')
print(f'{"Dimension":<20} {"Vorher":>10} {"Nachher":>10} {"Δ":>10}')
print('-' * 55)
for dim in before:
    delta = after[dim] - before[dim]
    print(f'{dim:<20} {before[dim]:>9.1f}% {after[dim]:>9.1f}% {delta:>+9.1f}%')

print(f'\n✅ Bereinigter Datensatz: {df_clean.shape}')
df_clean.to_csv("/tmp/verkaufsdaten_clean.csv", index=False)
print('Gespeichert als: verkaufsdaten_clean.csv')

## ✅ Challenges

1. Lade `customer_churn.csv` und behandle die fehlenden Werte in `total_charges`
2. Entferne alle Zeilen mit ungültigen Werten (z.B. negative Preise wenn vorhanden)
3. Erstelle einen SQL-Abfrage in SQLite: Wie viele saubere Datensätze gibt es?

In [ ]:
# Challenge 1:

# Challenge 2:

# Challenge 3:
import sqlite3
conn = sqlite3.connect("/tmp/dav_sample.db")
# deine SQL-Abfrage hier:
# df = pd.read_sql('SELECT ...', conn)
conn.close()


---
**Weiter mit:** `06_Daten_Transformieren.ipynb`